# AI Research Novelty & Gap Detector — Evaluation Notebook
**Authors:** PhD Researcher | **Institution:** Indian University

This notebook evaluates:
1. Novelty score accuracy vs expert judgment
2. Gap detection performance (precision/recall)
3. Retrieval quality (MRR, Precision@K)
4. Runtime efficiency
5. Visualisations for thesis inclusion

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from scipy.stats import pearsonr, spearmanr

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')
print('✅ Imports OK')

## 1. Load System Modules

In [ ]:
from retrieval.retriever import get_retriever
from models.novelty_engine import calculate_novelty_score
from models.similarity_engine import compute_similarity_stats, rank_results
from models.gap_engine import detect_research_gaps

retriever = get_retriever()
print(f'Retriever loaded. Corpus size: {retriever.get_corpus_size()}')

## 2. Synthetic Expert Judgment Dataset
We simulate expert novelty ratings for 20 test queries spanning HIGH/MEDIUM/LOW novelty.

In [ ]:
# Synthetic expert-labelled test cases
# In a real thesis study, these would be rated by 3–5 domain experts
EXPERT_TEST_CASES = [
    # HIGH novelty queries
    {'title': 'Quantum Consciousness and Vedic Philosophy: A Cross-Cultural Neuroscience Perspective',
     'abstract': 'This study explores intersections between quantum cognition theory and ancient Indian philosophical traditions.',
     'keywords': 'quantum consciousness, vedic philosophy, neuroscience',
     'expert_label': 'HIGH', 'expert_score': 88},
    {'title': 'Social Media Grief Rituals among Bereaved Youth in Coastal Tamil Nadu',
     'abstract': 'An ethnographic study of digital mourning practices among 18–25 year-olds in fishing communities.',
     'keywords': 'social media, grief, Tamil Nadu, youth, ethnography',
     'expert_label': 'HIGH', 'expert_score': 82},
    {'title': 'Algorithmic Bias in Indian Regional Language NLP Models',
     'abstract': 'Examines bias in pre-trained NLP models for Tamil, Telugu, and Kannada.',
     'keywords': 'NLP bias, regional languages, India, Tamil, Telugu',
     'expert_label': 'HIGH', 'expert_score': 85},
    {'title': 'Post-COVID Livelihood Transitions among Handloom Weavers in Varanasi',
     'abstract': 'Investigates occupational shifts and resilience strategies among artisan weavers post-pandemic.',
     'keywords': 'COVID-19, handloom, Varanasi, artisans, livelihood',
     'expert_label': 'HIGH', 'expert_score': 79},
    {'title': 'Intergenerational Trauma and Academic Performance among Scheduled Tribe Students',
     'abstract': 'Uses trauma-informed framework to examine educational outcomes among ST students in Jharkhand.',
     'keywords': 'intergenerational trauma, scheduled tribes, education, Jharkhand',
     'expert_label': 'HIGH', 'expert_score': 80},
    # MEDIUM novelty queries  
    {'title': 'Impact of Microfinance on Women Empowerment in Rural Maharashtra',
     'abstract': 'Studies self-help group outcomes on womens economic empowerment in Maharashtra villages.',
     'keywords': 'microfinance, women empowerment, Maharashtra, SHG',
     'expert_label': 'MEDIUM', 'expert_score': 55},
    {'title': 'Digital Literacy and Employment among Youth in Rajasthan',
     'abstract': 'Examines the relationship between digital skills and employment outcomes.',
     'keywords': 'digital literacy, youth, employment, Rajasthan',
     'expert_label': 'MEDIUM', 'expert_score': 58},
    {'title': 'Role of Panchayati Raj in Rural Development in Odisha',
     'abstract': 'Evaluates local governance effectiveness in rural development programs.',
     'keywords': 'panchayati raj, rural development, governance, Odisha',
     'expert_label': 'MEDIUM', 'expert_score': 52},
    {'title': 'Environmental Awareness among College Students in Bangalore',
     'abstract': 'Survey-based study of ecological consciousness among urban college youth.',
     'keywords': 'environment, college students, Bangalore, awareness',
     'expert_label': 'MEDIUM', 'expert_score': 60},
    {'title': 'Academic Stress and Mental Health among Engineering Students in Tamil Nadu',
     'abstract': 'Investigates academic pressure and psychological well-being.',
     'keywords': 'academic stress, mental health, engineering, Tamil Nadu',
     'expert_label': 'MEDIUM', 'expert_score': 50},
    # LOW novelty queries
    {'title': 'A Study on Livelihood Challenges among Farmers in Rural India',
     'abstract': 'Examines agricultural livelihoods and socio-economic challenges of rural farmers.',
     'keywords': 'livelihood, farmers, rural India, agriculture, poverty',
     'expert_label': 'LOW', 'expert_score': 22},
    {'title': 'Gender Inequality and Social Mobility among Women in India',
     'abstract': 'A study on gender discrimination and womens social mobility using survey methods.',
     'keywords': 'gender inequality, social mobility, women, India, survey',
     'expert_label': 'LOW', 'expert_score': 18},
    {'title': 'Political Participation and Social Capital in Urban India',
     'abstract': 'Quantitative study on voting behaviour and civic engagement in urban areas.',
     'keywords': 'political participation, social capital, urban India, voting',
     'expert_label': 'LOW', 'expert_score': 25},
    {'title': 'Educational Access for Scheduled Caste Students in Government Schools',
     'abstract': 'Investigates barriers and enablers for SC students in rural government schools.',
     'keywords': 'education, scheduled castes, government schools, access',
     'expert_label': 'LOW', 'expert_score': 20},
    {'title': 'Migration Patterns and Economic Development in North India',
     'abstract': 'Studies internal migration and its impact on household income in UP and Bihar.',
     'keywords': 'migration, economic development, Uttar Pradesh, Bihar',
     'expert_label': 'LOW', 'expert_score': 28},
]

print(f'Test cases: {len(EXPERT_TEST_CASES)}')
df_test = pd.DataFrame(EXPERT_TEST_CASES)
df_test['expert_label'].value_counts()

## 3. Run System on Test Cases

In [ ]:
results_log = []
runtimes    = []

for case in EXPERT_TEST_CASES:
    t0 = time.time()
    papers = retriever.retrieve(
        title=case['title'], abstract=case['abstract'],
        keywords=case['keywords'], top_k=10
    )
    sim_stats = compute_similarity_stats(papers)
    novelty   = calculate_novelty_score(papers, sim_stats)
    elapsed   = time.time() - t0
    runtimes.append(elapsed)
    
    results_log.append({
        'title':          case['title'][:50],
        'expert_label':   case['expert_label'],
        'expert_score':   case['expert_score'],
        'system_label':   novelty['label'],
        'system_score':   novelty['percentage'],
        'runtime_ms':     round(elapsed * 1000, 1),
    })

df_results = pd.DataFrame(results_log)
print('System evaluation complete.')
df_results.head(10)

## 4. Novelty Score Accuracy vs Expert Judgment

In [ ]:
# Label accuracy
label_match = (df_results['expert_label'] == df_results['system_label']).mean()

# Numeric correlation (expert_score vs system_score)
pearson_r, p_pearson = pearsonr(df_results['expert_score'], df_results['system_score'])
spearman_r, p_spear  = spearmanr(df_results['expert_score'], df_results['system_score'])
mae = mean_absolute_error(df_results['expert_score'], df_results['system_score'])
rmse= np.sqrt(mean_squared_error(df_results['expert_score'], df_results['system_score']))

print('='*50)
print('NOVELTY SCORE ACCURACY METRICS')
print('='*50)
print(f'Label Match Accuracy : {label_match:.1%}')
print(f'Pearson Correlation  : r={pearson_r:.3f}  (p={p_pearson:.4f})')
print(f'Spearman Correlation : ρ={spearman_r:.3f}  (p={p_spear:.4f})')
print(f'MAE (score diff)     : {mae:.1f} points')
print(f'RMSE                 : {rmse:.1f} points')

In [ ]:
# Classification report
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(['LOW', 'MEDIUM', 'HIGH'])
y_true = le.transform(df_results['expert_label'])
y_pred = le.transform(df_results['system_label'])

print(classification_report(y_true, y_pred, target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix: Expert vs System Labels', fontweight='bold')
ax.set_xlabel('System Prediction')
ax.set_ylabel('Expert Label')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix.png', dpi=150)
plt.show()
print('Saved confusion matrix.')

In [ ]:
# Scatter plot: Expert score vs System score
fig, ax = plt.subplots(figsize=(6,5))
colors = {'HIGH':'#2ecc71','MEDIUM':'#f39c12','LOW':'#e74c3c'}
for label in ['HIGH','MEDIUM','LOW']:
    sub = df_results[df_results['expert_label']==label]
    ax.scatter(sub['expert_score'], sub['system_score'],
               c=colors[label], label=label, s=80, alpha=0.8)

# Ideal line
x = np.linspace(0,100,100)
ax.plot(x, x, 'k--', alpha=0.4, label='Perfect Agreement')
ax.set_xlabel('Expert Novelty Score')
ax.set_ylabel('System Novelty Score')
ax.set_title(f'Expert vs System Scores (r={pearson_r:.2f})')
ax.legend()
ax.set_xlim(0,100); ax.set_ylim(0,100)
plt.tight_layout()
plt.savefig('../data/processed/score_correlation.png', dpi=150)
plt.show()

## 5. Gap Detection Performance
Simulate ground-truth gaps and evaluate detection accuracy.

In [ ]:
# Simulated ground truth for gap dimensions
# In a real study, these would be expert-verified
GROUND_TRUTH_GAPS = {
    'regional':    ['north-east india','assam','tribal regions','odisha'],
    'population':  ['transgender','disabled persons','elderly','widows'],
    'method':      ['longitudinal','grounded theory','action research','meta-analysis'],
    'thematic':    ['media','governance','disability','nutrition'],
}

# Run gap detection on a sample query
sample_papers = retriever.retrieve(
    title='Impact of Digital Literacy on Rural Women in India',
    abstract='Examines digital skill adoption and gender gaps in rural communities.',
    keywords='digital literacy, rural women, India, gender, education',
    top_k=10
)
gap_result = detect_research_gaps(sample_papers)
gap_dims   = gap_result.get('gap_dimensions', {})

# Precision & Recall per dimension
rows = []
for dim_key, gt_items in GROUND_TRUTH_GAPS.items():
    detected_key = f'{dim_key.rstrip("l")}al_gaps' if not dim_key.endswith('al') else f'{dim_key}_gaps'
    # Map keys
    key_map = {
        'regional': 'regional_gaps', 'population': 'population_gaps',
        'method': 'methodological_gaps', 'thematic': 'thematic_gaps'
    }
    detected = [g.lower() for g in gap_dims.get(key_map[dim_key], [])]
    gt       = [g.lower() for g in gt_items]

    tp = len(set(detected) & set(gt))
    fp = len(set(detected) - set(gt))
    fn = len(set(gt) - set(detected))
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    
    rows.append({'Dimension': dim_key.title(), 'Precision': prec,
                 'Recall': rec, 'F1': f1, 'TP': tp, 'FP': fp, 'FN': fn})

df_gap_eval = pd.DataFrame(rows)
print('Gap Detection Performance:')
print(df_gap_eval.to_string(index=False, float_format='{:.2f}'.format))

In [ ]:
# Bar chart of F1 per dimension
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(df_gap_eval))
w = 0.25
ax.bar(x - w, df_gap_eval['Precision'], w, label='Precision', color='#1565c0')
ax.bar(x,     df_gap_eval['Recall'],    w, label='Recall',    color='#2e7d32')
ax.bar(x + w, df_gap_eval['F1'],        w, label='F1 Score',  color='#f57c00')
ax.set_xticks(x)
ax.set_xticklabels(df_gap_eval['Dimension'])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Gap Detection: Precision / Recall / F1 by Dimension')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/gap_detection_metrics.png', dpi=150)
plt.show()

## 6. Retrieval Quality (Precision@K)

In [ ]:
# Evaluate Precision@K at various K values
# We define 'relevant' as papers with similarity >= 0.60
RELEVANCE_THRESHOLD = 0.60
K_VALUES = [1, 3, 5, 10]

def precision_at_k(papers, k, threshold=RELEVANCE_THRESHOLD):
    top_k = papers[:k]
    relevant = sum(1 for p in top_k if p.get('similarity_score', 0) >= threshold or p.get('similarity_pct', 0)/100.0 >= threshold)
    return relevant / k if k > 0 else 0

prec_results = {k: [] for k in K_VALUES}

for case in EXPERT_TEST_CASES:
    papers = retriever.retrieve(
        title=case['title'], abstract=case['abstract'],
        keywords=case['keywords'], top_k=10
    )
    for k in K_VALUES:
        prec_results[k].append(precision_at_k(papers, k))

print('Retrieval Quality — Precision@K')
print('-'*30)
for k in K_VALUES:
    avg = np.mean(prec_results[k])
    print(f'  P@{k:<3}: {avg:.3f}')

In [ ]:
# ── RRF Hybrid Comparison vs Dense vs Sparse ──

def retrieve_dense_only(query_text, top_k=10):
    query_vector = retriever.embedding_model.encode([query_text], convert_to_numpy=True)
    D, I = retriever.index.search(query_vector, top_k)
    results = []
    for rank, idx in enumerate(I[0]):
        if idx < len(retriever.metadata):
            p = retriever.metadata[idx].copy()
            p['similarity_score'] = float(D[0][rank])
            results.append(p)
    return results

def retrieve_sparse_only(query_text, top_k=10):
    tokens = query_text.lower().split()
    scores = retriever.bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_indices):
        p = retriever.metadata[idx].copy()
        p['similarity_score'] = float(scores[idx] / max(scores[idx], 1.0))
        results.append(p)
    return results

dense_p5 = []
sparse_p5 = []
hybrid_p5 = []

for case in EXPERT_TEST_CASES:
    query = f"{case['title']} {case['abstract']}"
    dense_res = retrieve_dense_only(query, top_k=5)
    sparse_res = retrieve_sparse_only(query, top_k=5)
    hybrid_res = retriever.retrieve(title=case['title'], abstract=case['abstract'], keywords=case['keywords'], top_k=5)
    
    dense_p5.append(precision_at_k(dense_res, 5))
    sparse_p5.append(precision_at_k(sparse_res, 5))
    hybrid_p5.append(precision_at_k(hybrid_res, 5))

print('Retrieval Engine Comparison — Mean Precision@5:')
print(f'  Dense-Only (FAISS)   : {np.mean(dense_p5):.3f}')
print(f'  Sparse-Only (BM25)   : {np.mean(sparse_p5):.3f}')
print(f'  Hybrid (RRF + CE)    : {np.mean(hybrid_p5):.3f}')

# Plot the retrieval comparison
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
engines = ['Dense-Only', 'Sparse-Only', 'Hybrid (RRF + CE)']
scores = [np.mean(dense_p5), np.mean(sparse_p5), np.mean(hybrid_p5)]
ax.bar(engines, scores, color=['#2980b9', '#27ae60', '#8e44ad'])
ax.set_ylabel('Mean Precision@5')
ax.set_title('Retrieval Effectiveness Comparison')
ax.set_ylim(0, 1.1)
for i, v in enumerate(scores):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/retrieval_comparison.png', dpi=150)
plt.show()

## 7. Runtime Efficiency

In [ ]:
print('Runtime Statistics (per query)')
print('-'*40)
rt = df_results['runtime_ms']
print(f'  Mean    : {rt.mean():.1f} ms')
print(f'  Median  : {rt.median():.1f} ms')
print(f'  Min     : {rt.min():.1f} ms')
print(f'  Max     : {rt.max():.1f} ms')
print(f'  Std Dev : {rt.std():.1f} ms')

fig, ax = plt.subplots(figsize=(7,3))
ax.bar(range(len(runtimes)), [r*1000 for r in runtimes], color='#1565c0')
ax.axhline(np.mean(runtimes)*1000, color='red', linestyle='--', label=f'Mean={np.mean(runtimes)*1000:.0f}ms')
ax.set_xlabel('Test Case #')
ax.set_ylabel('Runtime (ms)')
ax.set_title('Per-Query Runtime (Retrieval + Scoring)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/runtime_analysis.png', dpi=150)
plt.show()

## 8. Summary Table for Thesis

In [ ]:
summary = {
    'Metric': [
        'Novelty Label Accuracy', 'Pearson Correlation (r)',
        'Spearman Correlation (ρ)', 'MAE (score points)',
        'RMSE (score points)', 'Mean P@5',
        'Mean Query Latency (ms)', 'Corpus Size'
    ],
    'Value': [
        f'{label_match:.1%}', f'{pearson_r:.3f}',
        f'{spearman_r:.3f}', f'{mae:.1f}',
        f'{rmse:.1f}', f'{np.mean(prec_results[5]):.3f}',
        f'{np.mean(runtimes)*1000:.0f} ms',
        str(retriever.get_corpus_size())
    ]
}
df_summary = pd.DataFrame(summary)
print('\n=== THESIS EVALUATION SUMMARY ===')
print(df_summary.to_string(index=False))
df_summary.to_csv('../data/processed/evaluation_summary.csv', index=False)
print('\nSaved to data/processed/evaluation_summary.csv')